<a href="https://colab.research.google.com/github/jkp8330/assignment3-smtpclient/blob/main/final_vision_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ScienceQA Visual Challenge: Starter Notebook

This notebook provides a starting point for the ScienceQA Visual Multiple-Choice Challenge. It is based on the provided baseline solution, but has been adapted to be more of a general-purpose starter.

**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (~500 M params)
**Fine-Tuning:** QLoRA (4-bit NF4)
**Scoring:** Multiple-choice log-likelihood

---

In [2]:
# ── 0. Install libraries ──────────────────────────────────────────
# Run this cell to install the necessary Python packages.
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow
#!pip install --upgrade transformers peft bitsandbytes

In [3]:
import os

# 1. Define where your zip file lives in Drive and where you want it to go
drive_zip_path = "/content/pixels-to-predictions.zip"
local_extract_path = "/content/data"

# 2. Create the destination folder in the local Colab environment
os.makedirs(local_extract_path, exist_ok=True)

# 3. Use the bash 'unzip' command
# -q means "quiet" (suppresses output)
# -d specifies the destination directory
!unzip -q "{drive_zip_path}" -d "{local_extract_path}"

print("Unzipping complete!")

Unzipping complete!


In [4]:
# ── 1. Imports & Configuration ───────────────────────────────────────────────
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
# Adjust these paths to match your local environment
DATA_DIR   = Path("//content//data//pixels-to-predictions")

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# ── Basic Settings ───────────────────────────────────────────────────────────
IMG_SIZE        = 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: Tesla T4


## 2. Load and Preprocess Data

In [5]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

# The 'choices' column is a JSON string, so we parse it
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
train_df.head(2)

Train: 3,109 | Val: 1,048 | Test: 1,008


,id,image_path,question,choices,num_choices,answer,hint,lecture,solution,task,grade,subject,topic,category,skill
0,train_07667,images/train/train_07667.png,Why might putting each tadpole in its own pool...,[the male's tadpoles will be larger when they ...,3,2,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...
1,train_02628,images/train/train_02628.png,Why might forming strong social bonds with oth...,"[the female's offspring will live longer, the ...",3,0,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...


In [6]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=train_df)

https://docs.google.com/spreadsheets/d/1Rz8iMavgPS-gh4QbqV-eyx9AYPA9nVTQCtDLFyfIg3k/edit#gid=0


In [7]:
# ── 2b. Prompt Engineering ───────────────────────────────────────────────────
CHOICE_LETTERS = "ABCDEFGHIJ"

def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    """
    Builds the text prompt for the Vision Language Model.
    The <image> token is required for the model to process the image.
    """
    context_parts = []
    lecture = row.get("lecture", "")
    hint    = row.get("hint", "")
    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())
    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(
        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )

    prompt = "<image>\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        answer_idx = int(row['answer'])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt

# Display an example prompt
print(build_prompt(train_df.iloc[0], include_answer=True))

<image>
Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring that survive to reproduce. But the behaviors cannot guarantee that the animals will have greater reproductive success. Animals that attract or compete for mates won't always successful

In [8]:
# ── 2c. PyTorch Dataset ───────────────────────────────────────────────────────
class ScienceQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path, img_size: int = 224, is_train: bool = True):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size
        self.is_train = is_train

    def __len__(self) -> int:
        return len(self.df)

    def _load_image(self, rel_path: str) -> Image.Image:
        img = Image.open(self.data_dir / rel_path).convert("RGB")
        img = img.resize((self.img_size, self.img_size), Image.BICUBIC)
        return img

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        img = self._load_image(row["image_path"])

        if self.is_train:
            return {
                "image":  img,
                "text":   build_prompt(row, include_answer=True),
                "answer": int(row["answer"]),
            }
        else:
            return {
                "image":   img,
                "text":    build_prompt(row, include_answer=False),
                "choices": row["choices"],
                "answer":  int(row["answer"]) if "answer" in row else -1,
            }

train_ds = ScienceQADataset(train_df, DATA_DIR, img_size=IMG_SIZE, is_train=True)
val_ds   = ScienceQADataset(val_df,   DATA_DIR, img_size=IMG_SIZE, is_train=False)
test_ds  = ScienceQADataset(test_df,  DATA_DIR, img_size=IMG_SIZE, is_train=False)

print(f"Datasets created: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

Datasets created: train=3109, val=1048, test=1008


## 3. Model Loading and Inference Example

This section loads `HuggingFaceTB/SmolVLM-500M-Instruct` and runs a quick inference example on one validation sample.

In [9]:

# ── 3a. Load SmolVLM model + run one inference example ───────────────────────
from transformers import AutoProcessor, AutoModelForVision2Seq

processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
 )

if not torch.cuda.is_available():
    model.to(device)
model.eval()

# Pick a sample from validation set
sample = val_df.iloc[0]
sample_image = Image.open(DATA_DIR / sample["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
sample_prompt = build_prompt(sample, include_answer=False)

inputs = processor(
    text=[sample_prompt],
    images=[sample_image],
    return_tensors="pt",
)
inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
    )

decoded = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("Prompt:")
print(sample_prompt)
print("\nModel output:")
print(decoded)
print(f"\nGround-truth answer index: {sample['answer']}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Prompt:
<image>
Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring that survive to reproduce. But the behaviors cannot guarantee that the animals will have greater reproductive success. Animals that attract or compete for mates won't always su

In [10]:
# ── 4. Apply LoRA to meet < 5M trainable parameters rule ────────────────────
from peft import LoraConfig, get_peft_model

# SmolVLM utilizes a Llama-like text decoder.
# Targeting just q_proj and v_proj keeps parameters very low.
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Wrap the base model
model = get_peft_model(model, lora_config)

# Print parameters to verify compliance with the 5 million limit
model.print_trainable_parameters()

trainable params: 1,114,112 || all params: 508,596,416 || trainable%: 0.2191


In [11]:
# ── 5. Save Adapters for Offline Inference ──────────────────────────────────
import os

OUTPUT_DIR = "smolvlm_finetuned_adapters"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save only the lightweight LoRA weights, not the entire 500M model
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print(f"Adapters saved to {OUTPUT_DIR}. Upload this folder to your offline environment.")

Adapters saved to smolvlm_finetuned_adapters. Upload this folder to your offline environment.


In [12]:
# ── 6. Generate submission.csv (OPTIMIZED WITH BATCHING) ────────────────────
from tqdm import tqdm
import pandas as pd
import torch

# Ensure model is in evaluation mode
model.eval()

# Set batch size based on your GPU VRAM (Try 8 or 16. If you get an OOM error, lower it to 4)
BATCH_SIZE = 8

predictions = []
CHOICE_LETTERS = "ABCDEFGHIJ"

# 1. Setup Tokenizer for Batch Generation
# Generation requires left-padding so the new tokens are appended correctly
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "left"

print(f"Running batched inference on {len(test_ds)} samples (Batch Size: {BATCH_SIZE})...")

# 2. Iterate through the dataset in chunks
for batch_start in tqdm(range(0, len(test_ds), BATCH_SIZE)):
    batch_end = min(batch_start + BATCH_SIZE, len(test_ds))

    # Slice the dataframe to get the corresponding IDs
    batch_ids = test_df.iloc[batch_start:batch_end]["id"].tolist()

    # Extract images and texts for this batch
    batch_samples = [test_ds[i] for i in range(batch_start, batch_end)]
    texts = [sample["text"] for sample in batch_samples]
    images = [sample["image"] for sample in batch_samples]

    # Process the entire batch at once
    inputs = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True
    )
    # Move all input tensors to the GPU
    inputs = {k: v.to(device) if torch.is_tensor(v) else v for k, v in inputs.items()}

    # Generate answers for the batch
    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=5,       # Keep it short for MCQs
            do_sample=False,        # Greedy decoding for consistent answers
            pad_token_id=processor.tokenizer.pad_token_id
        )

    # 3. Decode the batch safely
    # We must slice off the input tokens to only look at what the model generated
    input_len = inputs["input_ids"].shape[1]

    for i, gen_id in enumerate(generated_ids):
        # Extract just the newly generated tokens
        new_tokens = gen_id[input_len:]
        predicted_text = processor.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        # Map the model's letter output (e.g., 'A', 'B') back to a 0-indexed integer
        pred_idx = 0 # Default fallback if the model hallucinates
        if predicted_text and predicted_text[0] in CHOICE_LETTERS:
            pred_idx = CHOICE_LETTERS.index(predicted_text[0])

        predictions.append({
            "id": batch_ids[i],
            "answer": pred_idx
        })

# Export strictly to id and answer columns
submission_df = pd.DataFrame(predictions)
submission_df.to_csv("submission.csv", index=False)
print("Saved submission.csv successfully!")

Running batched inference on 1008 samples (Batch Size: 8)...


100%|██████████| 126/126 [18:33<00:00,  8.83s/it]

Saved submission.csv successfully!
